Basic imports, configuration constants, and MONAI setup following the official tutorial style.

In [1]:
import os
import numpy as np
from pathlib import Path

import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn

from monai.data import ImageDataset, CacheDataset, Dataset
from monai.transforms import (
    LoadImage,
    EnsureChannelFirst,
    Resize,
    ScaleIntensity,
    ToTensor,
    Compose,
    RandFlip,
    RandRotate,
)
from monai.networks.nets import resnet50
from monai.metrics import ROCAUCMetric
from monai.losses import FocalLoss
from monai.utils import set_determinism

set_determinism(42)

DATA_DIR = Path("../dataset/331")
CLASS_NAMES = ["Normal", "DDH"]
IMAGE_SIZE = 331
BATCH_SIZE = 16
EPOCHS = 10
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15


IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Load all file paths from the dataset folders and build the label list in MONAI-compatible format.

In [2]:
image_files = []
image_labels = []

for idx, cname in enumerate(CLASS_NAMES):
    cdir = DATA_DIR / cname
    files = sorted([str(p) for p in cdir.glob("*") if p.is_file()])
    image_files.extend(files)
    image_labels.extend([idx] * len(files))

num_images = len(image_files)
num_images


314

Create MONAI-compatible train/val/test partitions using random_split with your configured ratios.

In [3]:
total = len(image_files)
val_count = int(total * VAL_SPLIT)
test_count = int(total * TEST_SPLIT)
train_count = total - val_count - test_count

train_files, val_files, test_files = torch.utils.data.random_split(
    list(range(total)),
    [train_count, val_count, test_count],
    generator=torch.Generator().manual_seed(42)
)

train_images = [image_files[i] for i in train_files.indices]
train_labels = [image_labels[i] for i in train_files.indices]

val_images = [image_files[i] for i in val_files.indices]
val_labels = [image_labels[i] for i in val_files.indices]

test_images = [image_files[i] for i in test_files.indices]
test_labels = [image_labels[i] for i in test_files.indices]

len(train_images), len(val_images), len(test_images)


(220, 47, 47)

Basic preprocessing (load → channel-first → resize → intensity scaling → tensor).
Light augmentation only on training (flip + rotation).
Kept minimal, consistent with MONAI tutorials.

In [16]:
train_data = [{"img": f, "label": l} for f, l in zip(train_images, train_labels)]
val_data   = [{"img": f, "label": l} for f, l in zip(val_images, val_labels)]
test_data  = [{"img": f, "label": l} for f, l in zip(test_images, test_labels)]

from monai.transforms import (
    LoadImageD,
    EnsureChannelFirstD,
    ResizeD,
    ScaleIntensityD,
    RandFlipD,
    RandRotateD,
    ToTensorD,
)

train_transforms = Compose([
    LoadImageD(keys="img"),
    EnsureChannelFirstD(keys="img"),
    ResizeD(keys="img", spatial_size=(IMAGE_SIZE, IMAGE_SIZE)),
    ScaleIntensityD(keys="img"),
    RandFlipD(keys="img", prob=0.5, spatial_axis=1),
    RandRotateD(keys="img", range_x=15, prob=0.5),
    ToTensorD(keys=["img", "label"]),
])

val_transforms = Compose([
    LoadImageD(keys="img"),
    EnsureChannelFirstD(keys="img"),
    ResizeD(keys="img", spatial_size=(IMAGE_SIZE, IMAGE_SIZE)),
    ScaleIntensityD(keys="img"),
    ToTensorD(keys=["img", "label"]),
])

train_ds = Dataset(train_data, train_transforms)
val_ds = Dataset(val_data, val_transforms)
test_ds = Dataset(test_data, val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)



Wrap the image paths + labels with MONAI ImageDataset and create PyTorch dataloaders for training, validation, and testing.

In [17]:
train_ds = ImageDataset(
    image_files=train_images,
    labels=train_labels,
    transform=train_transforms
)

val_ds = ImageDataset(
    image_files=val_images,
    labels=val_labels,
    transform=val_transforms
)

test_ds = ImageDataset(
    image_files=test_images,
    labels=test_labels,
    transform=test_transforms
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

len(train_ds), len(val_ds), len(test_ds)


(220, 47, 47)

Instantiate a MONAI ResNet50 model for 2-class classification.
Output layer = 1 unit for binary classification (with BCE-style loss).

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet50(
    spatial_dims=2,
    n_input_channels=3,
    num_classes=1
).to(device)


Use weighted BCE loss for class imbalance and standard Adam optimizer.
Metric = ROC-AUC, commonly used in medical imaging binary classification.

In [19]:
pos_weight = torch.tensor([len(train_labels) / sum(train_labels)]).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

auc_metric = ROCAUCMetric()


A clean PyTorch loop consistent with MONAI tutorials:
forward → compute loss → backward → update → evaluate on val set each epoch.

In [20]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_ds)

    model.eval()
    val_outputs = []
    val_labels_all = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            val_outputs.append(outputs)
            val_labels_all.append(labels)

        val_outputs = torch.cat(val_outputs, dim=0)
        val_labels_all = torch.cat(val_labels_all, dim=0)
        val_auc = auc_metric(y_pred=val_outputs, y=val_labels_all).item()
        auc_metric.reset()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f} - Val AUC: {val_auc:.4f}")


RuntimeError: applying transform <monai.transforms.compose.Compose object at 0x13f5c64d0>

In [14]:
bad = [x for x in train_images if not isinstance(x, str)]
len(bad), bad[:3]

train_images[:5]



['../dataset/331/Normal/282.jpg',
 '../dataset/331/Normal/194.jpg',
 '../dataset/331/DDH/4.jpg',
 '../dataset/331/Normal/186.jpg',
 '../dataset/331/Normal/91.jpg']

In [ ]:
types = list({type(x) for x in train_labels})


[int]